In [1]:
import pandas as pd


In [2]:
# Load cleaned datasets
admissions = pd.read_csv("admissions_clean.csv")
doctors = pd.read_csv("doctors_preprocessed.csv")
departments = pd.read_csv("departments_clean.csv")
beds = pd.read_csv("bed_capacity.csv")   # NEW dataset for Milestone 3


In [3]:
# Rename columns for consistency
departments.rename(columns={"department_id": "Department_ID"}, inplace=True)
beds.rename(columns={"department_id": "Department_ID", "Total_Beds": "Bed_Capacity"}, inplace=True)


In [4]:
# Count admissions per department
admissions_count = admissions.groupby("Department_ID").size().reset_index(name="Admissions")




In [5]:
# Count doctors per department
doctors_count = doctors.groupby("Department_ID").size().reset_index(name="Doctors")


In [6]:
# Merge admissions, doctors, departments, and bed capacity
# Drop department_name from beds to avoid duplicate
dept_util = (
    admissions_count
    .merge(doctors_count, on="Department_ID")
    .merge(departments, on="Department_ID")   # keep department_name here
    .merge(beds.drop(columns=["department_name"]), on="Department_ID")  # drop duplicate
)


In [7]:
# Patients per doctor and bed utilization
dept_util["Patients_per_Doctor"] = dept_util["Admissions"] / dept_util["Doctors"]
dept_util["Bed_Utilization"] = dept_util["Admissions"] / dept_util["Bed_Capacity"]


In [8]:
# Weighted average of doctor workload and bed utilization
# Normalizing doctor workload: assume 20 patients/doctor = 100%
dept_util["Resource_Utilization_Score"] = (
    0.5 * (dept_util["Patients_per_Doctor"] / 20) +
    0.5 * dept_util["Bed_Utilization"]
)


In [9]:
# Classify departments based on score thresholds
def classify(row):
    if row["Resource_Utilization_Score"] > 0.85:
        return "Overloaded"
    elif row["Resource_Utilization_Score"] < 0.5:
        return "Underutilized"
    else:
        return "Balanced"

dept_util["Status"] = dept_util.apply(classify, axis=1)


In [11]:
# Check column names of the final table
print(dept_util.columns)


Index(['Department_ID', 'Admissions', 'Doctors', 'department_name',
       'Bed_Capacity', 'Patients_per_Doctor', 'Bed_Utilization',
       'Resource_Utilization_Score', 'Status'],
      dtype='str')


In [15]:
# Show first 10 rows
dept_util.head(10)


,Department_ID,Admissions,Doctors,department_name,Bed_Capacity,Patients_per_Doctor,Bed_Utilization,Resource_Utilization_Score,Status,Doctor_Workload_Index,Bed_Utilization_Index
0,D001,233,23,Cardiology,40,10.130435,5.825000,0.572872,Underutilized,0.926759,0.218985
1,D002,200,20,Neurology,30,10.000000,6.666667,0.582727,Underutilized,0.914826,0.250627
2,D003,320,30,Orthopedics,35,10.666667,9.142857,0.659766,Balanced,0.975815,0.343716
3,D004,277,27,Pediatrics,45,10.259259,6.155556,0.584978,Underutilized,0.938544,0.231412
4,D005,331,33,Oncology,35,10.030303,9.457143,0.636565,Balanced,0.917599,0.355532
5,D006,215,21,ENT,15,10.238095,14.333333,0.737728,Overloaded,0.936608,0.538847
6,D007,179,20,Dermatology,10,8.950000,17.900000,0.745851,Overloaded,0.818770,0.672932
7,D008,175,18,General Surgery,40,9.722222,4.375000,0.526944,Underutilized,0.889415,0.164474
8,D009,169,17,Urology,20,9.941176,8.450000,0.613557,Balanced,0.909445,0.317669
9,D010,218,22,Nephrology,25,9.909091,8.720000,0.617165,Balanced,0.906510,0.327820


In [14]:
# Normalize metrics relative to dataset
dept_util["Doctor_Workload_Index"] = dept_util["Patients_per_Doctor"] / dept_util["Patients_per_Doctor"].max()
dept_util["Bed_Utilization_Index"] = dept_util["Bed_Utilization"] / dept_util["Bed_Utilization"].max()

# Combined score
dept_util["Resource_Utilization_Score"] = (
    0.5 * dept_util["Doctor_Workload_Index"] +
    0.5 * dept_util["Bed_Utilization_Index"]
)

# Percentile thresholds to spread categories
low = dept_util["Resource_Utilization_Score"].quantile(0.33)
high = dept_util["Resource_Utilization_Score"].quantile(0.66)

def classify(row):
    if row["Resource_Utilization_Score"] >= high:
        return "Overloaded"
    elif row["Resource_Utilization_Score"] <= low:
        return "Underutilized"
    else:
        return "Balanced"

dept_util["Status"] = dept_util.apply(classify, axis=1)


In [ ]:
# Save final table to CSV
dept_util.to_csv("updated_department_resource_utilization.csv", index=False)